# 🧪 Molecular Property Prediction — Hybrid GIN + ChemBERTa Fusion

This notebook trains a **hybrid molecular property predictor** that fuses:
- **GIN** (Graph Isomorphism Network) graph-level representations
- **ChemBERTa** text embeddings (dual-pooling)

into a gated bilinear fusion model for aligned multitask regression on **QM9**
quantum-mechanical properties.

---

### Notebook Structure

| # | Section | Purpose |
|---|---------|----------|
| 1 | Environment Setup | Detect cloud runtime, install deps, configure device |
| 2 | Imports & Configuration | All imports, paths, and hyperparameters |
| 3 | Model & Dataset Definitions | `HybridDataset`, `HybridFusionModel`, loss functions |
| 4 | Preprocessing — GIN Graphs | Build / load cached PyG graph artifacts |
| 5 | Preprocessing — ChemBERTa Tokenisation | Build / load cached tokenised text artifacts |
| 6 | Data Loading & Alignment | Align multimodal data, create DataLoaders |
| 7 | Training Loop | Train with early stopping, per-epoch metrics |
| 8 | Results Summary | Final per-property breakdown |

---
## 1 · Environment Setup

Detect whether we are running on **Google Colab**, **Kaggle**, **SageMaker**,
or a **local** machine.  Install missing dependencies and clone the repo if
needed.

In [ ]:
import os, sys, subprocess, shutil

# ── Detect cloud runtime ──────────────────────────────────────────────
IS_COLAB   = "google.colab" in str(getattr(sys, 'modules', {}))
IS_KAGGLE  = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
IS_SAGEMAKER = os.environ.get("SM_NUM_GPUS") is not None
IS_CLOUD   = IS_COLAB or IS_KAGGLE or IS_SAGEMAKER

RUNTIME = (
    "Google Colab" if IS_COLAB else
    "Kaggle"       if IS_KAGGLE else
    "SageMaker"    if IS_SAGEMAKER else
    "Local / Other"
)
print(f"🖥️  Runtime detected: {RUNTIME}")

# ── Clone repository if running in the cloud ──────────────────────────
REPO_URL  = "https://github.com/TETRAWasTaken/Molecular-Property-Prediction-Using-GNN-and-Transformer.git"
REPO_DIR  = "Molecular-Property-Prediction-Using-GNN-and-Transformer"

if IS_CLOUD and not os.path.isdir(REPO_DIR):
    print(f"📥 Cloning repository into ./{REPO_DIR} …")
    subprocess.check_call(["git", "clone", REPO_URL])

if IS_CLOUD:
    os.chdir(REPO_DIR)
    # Ensure the repo root is on sys.path so local imports resolve
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f"📂 Working directory: {os.getcwd()}")

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────
# Only runs if a key package is missing — avoids re-installing every run.

def _is_importable(name: str) -> bool:
    try:
        __import__(name)
        return True
    except ImportError:
        return False

REQUIRED = [
    ("torch",           "torch"),
    ("torch_geometric", "torch-geometric"),
    ("transformers",    "transformers"),
    ("rdkit",           "rdkit"),
    ("sklearn",         "scikit-learn"),
    ("scipy",           "scipy"),
    ("pandas",          "pandas"),
    ("matplotlib",      "matplotlib"),
    ("networkx",        "networkx"),
    ("art",             "art"),
]

missing = [pip_name for imp_name, pip_name in REQUIRED if not _is_importable(imp_name)]

if missing:
    print(f"📦 Installing missing packages: {', '.join(missing)}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + missing
    )
    print("✅ Installation complete.")
else:
    print("✅ All required packages already installed.")

In [ ]:
# ── Device configuration ──────────────────────────────────────────────
import multiprocessing, torch

os.environ.setdefault("HF_HUB_OFFLINE", "1")

# Use all CPU cores for intra-op parallelism
NUM_CPUS = multiprocessing.cpu_count()
torch.set_num_threads(NUM_CPUS)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"🚀 GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("🍎 Apple Silicon MPS backend detected.")
else:
    DEVICE = torch.device("cpu")
    print(f"⚠️  No GPU detected — running on CPU ({NUM_CPUS} threads).")

print(f"📟 Device: {DEVICE}")
print(f"🔧 PyTorch version: {torch.__version__}")

---
## 2 · Imports & Configuration

In [ ]:
import torch.nn as nn
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from torch.utils.data import Dataset, random_split
from torch_geometric.loader import DataLoader

from GIN_2.Utils.GIN import GIN
from Transformers_2.Utils.Transformer import StandaloneChemBERTa

print("✅ All imports successful.")

In [ ]:
# ── Paths (override via env vars for different cloud setups) ─────────
MOLECULE_CSV_PATH      = os.environ.get("MOLECULE_CSV_PATH",      "Dataset/New_QM9/molecule_properties.csv")
ATOM_CSV_PATH          = os.environ.get("ATOM_CSV_PATH",          "Dataset/New_QM9/atom_properties.csv")
TOKENIZED_CACHE_PATH   = os.environ.get("TOKENIZED_CACHE_PATH",   "Transformers_2/outputs/cache/tokenized_dataset.pt")
HYBRID_MODEL_OUTPUT_PATH = os.environ.get("HYBRID_MODEL_OUTPUT_PATH", "best_hybrid_model.pth")

# ── Target properties ─────────────────────────────────────────────────
TARGET_COLS = ['mu', 'alpha', 'homo', 'lumo', 'gap', 'r2', 'zpve', 'u0', 'u298', 'h298', 'g298', 'cv']

# ── Hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE              = 64
EPOCHS                  = 60
FREEZE_TRANSFORMER_EP   = 10    # epochs to keep transformer body frozen
PATIENCE                = 10    # early-stopping patience
SEED                    = 42

print("📋 Configuration")
print(f"   Targets         : {len(TARGET_COLS)} properties")
print(f"   Batch size      : {BATCH_SIZE}")
print(f"   Max epochs      : {EPOCHS}")
print(f"   Patience        : {PATIENCE}")
print(f"   Freeze epochs   : {FREEZE_TRANSFORMER_EP}")

---
## 3 · Model & Dataset Definitions

In [ ]:
class HybridDataset(Dataset):
    """Dataset wrapper that keeps graph, token, target, and mask tensors aligned.

    Each item exposes the PyG graph object together with the corresponding
    tokenized text inputs, regression targets, and NaN mask used to ignore
    missing labels during loss computation.
    """

    def __init__(self, pyg_graph_list, tokenized_input_ids, tokenized_attention_masks, targets, nan_mask):
        self.graphs = pyg_graph_list
        self.input_ids = tokenized_input_ids
        self.attention_masks = tokenized_attention_masks
        self.targets = targets
        self.nan_mask = nan_mask

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        return {
            'graph': self.graphs[idx],
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'target': self.targets[idx],
            'nan_mask': self.nan_mask[idx]
        }

In [ ]:
class HybridFusionModel(nn.Module):
    """Fuse graph and text embeddings into a multitask molecular property head.

    The model encodes molecular graphs with a GIN backbone, encodes molecular
    text with ChemBERTa using Dual Pooling, projects the text embedding into
    the graph embedding space, applies learned gating to both modalities,
    constructs an explicit bilinear interaction, and predicts all target
    properties with a shared MLP.
    """

    def __init__(
        self,
        gin_hidden_dim: int = 512,
        transformer_model: str = "seyonec/ChemBERTa-zinc-base-v1",
        mlp_hidden_dim: int = 1024,
        output_dim: int = 12,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.graph_encoder = GIN(hidden_dim=gin_hidden_dim, output_dim=output_dim)
        self.text_encoder = StandaloneChemBERTa(
            model_name=transformer_model, num_targets=output_dim
        )

        self.text_projector = nn.Sequential(
            nn.Linear(self.text_encoder.pooled_hidden_size, gin_hidden_dim),
            nn.BatchNorm1d(gin_hidden_dim),
            nn.ReLU(),
        )

        self.graph_gate = nn.Sequential(
            nn.Linear(gin_hidden_dim, gin_hidden_dim), nn.Sigmoid()
        )
        self.text_gate = nn.Sequential(
            nn.Linear(gin_hidden_dim, gin_hidden_dim), nn.Sigmoid()
        )

        self.bilinear = nn.Bilinear(gin_hidden_dim, gin_hidden_dim, gin_hidden_dim)

        # Strip prediction heads: we only want embeddings from sub-encoders.
        self.graph_encoder.prediction_head[-1] = nn.Identity()
        self.text_encoder.prediction_head = nn.Identity()

        concat_dim = gin_hidden_dim * 3

        self.fusion_mlp = nn.Sequential(
            nn.Linear(concat_dim, mlp_hidden_dim),
            nn.BatchNorm1d(mlp_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, mlp_hidden_dim // 2),
            nn.BatchNorm1d(mlp_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim // 2, output_dim),
        )

    def forward(self, graph_data, input_ids, attention_mask, num_graphs=None):
        graph_embedding = self.graph_encoder(graph_data, num_graphs=num_graphs)
        raw_text_embedding = self.text_encoder(input_ids, attention_mask)
        text_embedding = self.text_projector(raw_text_embedding)

        g_weight = self.graph_gate(graph_embedding)
        t_weight = self.text_gate(text_embedding)

        weighted_graph = graph_embedding * g_weight
        weighted_text  = text_embedding * t_weight

        interaction = torch.relu(self.bilinear(weighted_graph, weighted_text))
        fused_embedding = torch.cat(
            [weighted_graph, weighted_text, interaction], dim=1
        )
        return self.fusion_mlp(fused_embedding)

print("✅ HybridFusionModel defined.")

In [ ]:
def masked_mse_loss(predictions, targets, nan_mask, beta=1.0):
    """NaN-masked SmoothL1 regression loss for validation monitoring."""
    loss_fn = nn.SmoothL1Loss(reduction='none', beta=beta)
    raw_loss = loss_fn(predictions, targets)
    masked_loss = raw_loss * nan_mask
    valid_entries = nan_mask.sum()
    if valid_entries > 0:
        return masked_loss.sum() / valid_entries
    return torch.tensor(0.0, device=predictions.device, requires_grad=True)


class UncertaintyWeightedLoss(nn.Module):
    """Homoscedastic uncertainty-weighted multitask loss (Kendall et al. 2018).

    Learns a per-task log-variance and automatically down-weights
    high-uncertainty tasks.
    """

    def __init__(self, num_tasks: int) -> None:
        super().__init__()
        self.log_var = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, predictions, targets, nan_mask):
        loss_fn = nn.SmoothL1Loss(reduction='none', beta=1.0)
        raw_loss = loss_fn(predictions, targets)

        valid_counts = nan_mask.sum(0).clamp(min=1.0)
        per_task_loss = (raw_loss * nan_mask).sum(0) / valid_counts

        precision = torch.exp(-self.log_var)
        return (per_task_loss * precision * 0.5 + self.log_var * 0.5).sum()

print("✅ Loss functions defined.")

---
## 4 · Preprocessing — GIN Graphs

Build the PyG graph dataset from the QM9 CSVs. This step is **skipped** if
the processed cache already exists on disk.

In [ ]:
%%time

GIN_CACHE_FILE = os.path.join('GIN_2/data/processed', 'qm_merged_3d_graphs_delta.pt')
FORCE_REBUILD  = os.environ.get("FORCE_REBUILD", "0") == "1"

need_gin = FORCE_REBUILD or not os.path.exists(GIN_CACHE_FILE)

if need_gin:
    print("⏳ GIN graph cache not found — building from CSVs …")
    # Verify source files exist before spending time on preprocessing
    for fpath, label in [(MOLECULE_CSV_PATH, "Molecule CSV"), (ATOM_CSV_PATH, "Atom CSV")]:
        if not os.path.exists(fpath):
            raise FileNotFoundError(
                f"{label} not found at '{fpath}'. "
                f"Set the MOLECULE_CSV_PATH / ATOM_CSV_PATH env vars or "
                f"upload the dataset to the expected location."
            )
    from GIN_2.Utils.preprocessing import RelationalGeometryPipeline

    _ = RelationalGeometryPipeline(
        root='GIN_2/data',
        mol_csv_path=MOLECULE_CSV_PATH,
        atom_csv_path=ATOM_CSV_PATH,
        target_cols=TARGET_COLS,
    )
    print("✅ GIN preprocessing complete.")
else:
    print(f"✅ GIN cache found at '{GIN_CACHE_FILE}' — skipping preprocessing.")

---
## 5 · Preprocessing — ChemBERTa Tokenisation

Tokenise SMILES strings with the ChemBERTa tokeniser. Cached to disk to
avoid repeated work.

In [ ]:
%%time

need_transformer = FORCE_REBUILD or not os.path.exists(TOKENIZED_CACHE_PATH)

if need_transformer:
    print("⏳ Tokenised cache not found — running ChemBERTa tokeniser …")
    if not os.path.exists(MOLECULE_CSV_PATH):
        raise FileNotFoundError(
            f"Molecule CSV not found at '{MOLECULE_CSV_PATH}'. "
            f"Set the MOLECULE_CSV_PATH env var or upload the file."
        )
    from Transformers_2.Utils.Tokeniser import Tokeniser

    tokeniser = Tokeniser(
        mol_path=MOLECULE_CSV_PATH,
        model_name="seyonec/ChemBERTa-zinc-base-v1",
        max_length=64,
        use_cache=True,
        cache_path=TOKENIZED_CACHE_PATH,
    )
    tokeniser.run_tokenizer(verbose=False)
    print("✅ Transformer tokenisation complete.")
else:
    print(f"✅ Tokenised cache found at '{TOKENIZED_CACHE_PATH}' — skipping.")

---
## 6 · Data Loading & Alignment

Load both cached artifacts, align them by `mol_id`, and split into
train / val / test sets.

In [ ]:
%%time

print("📂 Loading cached datasets from disk …")

from GIN_2.Utils.preprocessing import RelationalGeometryPipeline

pyg_dataset = RelationalGeometryPipeline(
    root='GIN_2/data',
    mol_csv_path=MOLECULE_CSV_PATH,
    atom_csv_path=ATOM_CSV_PATH,
    target_cols=TARGET_COLS,
)
pyg_graph_list = list(pyg_dataset)
print(f"   GIN graphs loaded: {len(pyg_graph_list):,}")

transformer_data = torch.load(TOKENIZED_CACHE_PATH, weights_only=False)
print(f"   Transformer tokens loaded (keys: {list(transformer_data.keys())})")

In [ ]:
# ── Build mol_id → graph lookup ──────────────────────────────────────
graph_dict = {}
for g in pyg_graph_list:
    clean_id = str(g.mol_id.item()) if torch.is_tensor(g.mol_id) else str(g.mol_id)
    graph_dict[clean_id] = g

t_input_ids       = transformer_data['input_ids']
t_attention_masks  = transformer_data['attention_mask']
t_targets         = transformer_data['labels']
t_nan_mask        = transformer_data['nan_mask']
t_mol_ids         = [str(m).strip() for m in transformer_data['mol_ids']]
t_scalers         = transformer_data['scalers']

# ── Sanity-check ID formats ──────────────────────────────────────────
sample_t_id = t_mol_ids[0]
sample_g_id = next(iter(graph_dict))
print(f"  Sample Transformer ID : '{sample_t_id}' (type={type(sample_t_id).__name__})")
print(f"  Sample Graph ID      : '{sample_g_id}' (type={type(sample_g_id).__name__})")

if sample_t_id not in graph_dict and sample_g_id not in set(t_mol_ids[:100]):
    print("⚠️  WARNING: Sample IDs don't appear to match — alignment may fail!")
    print("   Check that mol_id formats are consistent between GIN and Transformer caches.")

In [ ]:
# ── Align multimodal data ────────────────────────────────────────────
print("🔗 Aligning multimodal data …")

aligned_graphs          = []
aligned_input_ids       = []
aligned_attention_masks  = []
aligned_targets         = []
aligned_nan_masks       = []

for i, mol_id in enumerate(t_mol_ids):
    if mol_id in graph_dict:
        aligned_graphs.append(graph_dict[mol_id])
        aligned_input_ids.append(t_input_ids[i])
        aligned_attention_masks.append(t_attention_masks[i])
        aligned_targets.append(t_targets[i])
        aligned_nan_masks.append(t_nan_mask[i])

if len(aligned_input_ids) == 0:
    raise ValueError(
        "Zero molecules were aligned!  Check that 'mol_id' in Graph objects "
        "matches the IDs in 'tokenized_dataset.pt'."
    )

n_total   = len(t_mol_ids)
n_aligned = len(aligned_graphs)
print(f"✅ Alignment complete: {n_aligned:,} / {n_total:,} molecules kept ({n_aligned/n_total*100:.1f}%)")

if n_aligned < n_total * 0.9:
    print(f"⚠️  WARNING: More than 10% of molecules were dropped during alignment.")

aligned_input_ids       = torch.stack(aligned_input_ids)
aligned_attention_masks  = torch.stack(aligned_attention_masks)
aligned_targets         = torch.stack(aligned_targets)
aligned_nan_masks       = torch.stack(aligned_nan_masks)

In [ ]:
# ── Create Dataset & DataLoaders ─────────────────────────────────────
full_dataset = HybridDataset(
    aligned_graphs,
    aligned_input_ids,
    aligned_attention_masks,
    aligned_targets,
    aligned_nan_masks,
)

train_size = int(0.8 * len(full_dataset))
val_size   = int(0.1 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)

num_workers = min(NUM_CPUS, 8)
# Colab / Kaggle sometimes misbehave with multiprocessing DataLoader workers
if IS_CLOUD:
    num_workers = min(num_workers, 4)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=num_workers)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)

print(f"\n📊 Dataset splits")
print(f"   Train : {train_size:>7,} samples  ({train_size/len(full_dataset)*100:.0f}%)")
print(f"   Val   : {val_size:>7,} samples  ({val_size/len(full_dataset)*100:.0f}%)")
print(f"   Test  : {test_size:>7,} samples  ({test_size/len(full_dataset)*100:.0f}%)")
print(f"   Workers: {num_workers}")

---
## 7 · Training Loop

Train with:
- **Uncertainty-weighted** multitask loss (learnable per-task precision)
- **Cosine annealing** with warm restarts
- **Gradient clipping** (max_norm=1.0)
- **Early stopping** with patience
- **Transformer freezing** for the first N epochs

In [ ]:
# ── Instantiate model, loss, optimiser, scheduler ────────────────────
model = HybridFusionModel().to(DEVICE)

train_loss_fn = UncertaintyWeightedLoss(len(TARGET_COLS)).to(DEVICE)

# Layer-wise LR: low LR for the pretrained transformer body
transformer_body_params = list(model.text_encoder.transformer.parameters())
non_transformer_text_params = [
    p for n, p in model.text_encoder.named_parameters()
    if 'transformer' not in n
]

optimizer = torch.optim.AdamW([
    {'params': model.graph_encoder.parameters(),  'lr': 3e-4, 'weight_decay': 1e-3},
    {'params': model.fusion_mlp.parameters(),     'lr': 3e-4, 'weight_decay': 1e-2},
    {'params': model.text_projector.parameters(), 'lr': 3e-4, 'weight_decay': 1e-3},
    {'params': model.graph_gate.parameters(),     'lr': 3e-4, 'weight_decay': 1e-3},
    {'params': model.text_gate.parameters(),      'lr': 3e-4, 'weight_decay': 1e-3},
    {'params': non_transformer_text_params,       'lr': 3e-4, 'weight_decay': 1e-5},
    {'params': transformer_body_params,           'lr': 1e-5, 'weight_decay': 1e-5},
    {'params': train_loss_fn.parameters(),        'lr': 1e-3, 'weight_decay': 0.0},
])

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Count trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n🧠 Model created")
print(f"   Total params     : {total_params:>12,}")
print(f"   Trainable params : {trainable_params:>12,}")
print(f"   Device           : {DEVICE}")

In [ ]:
# ── Training & Validation Loop ───────────────────────────────────────
from IPython.display import clear_output, display, HTML
import time

early_stop_counter = 0
best_val_loss      = float('inf')
history            = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_r2': []}

print(f"\n🏋️ Starting Hybrid Training on {DEVICE}")
print(f"   Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  Patience: {PATIENCE}")
print("=" * 90)

training_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # ── Freeze / unfreeze transformer body ────────────────────────────
    freeze = epoch < FREEZE_TRANSFORMER_EP
    for param in model.text_encoder.transformer.parameters():
        param.requires_grad = not freeze

    # ── Train ─────────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        if batch_idx % 100 == 0:
            print(
                f"\r  Epoch {epoch+1:02d}/{EPOCHS} | "
                f"Train batch {batch_idx}/{len(train_loader)} "
                f"{'[transformer frozen]' if freeze else ''}",
                end="", flush=True,
            )

        graph_data      = batch['graph'].to(DEVICE)
        b_input_ids     = batch['input_ids'].to(DEVICE)
        b_attention_mask = batch['attention_mask'].to(DEVICE)
        b_targets       = batch['target'].to(DEVICE)
        b_nan_mask      = batch['nan_mask'].to(DEVICE)

        optimizer.zero_grad()
        predictions = model(graph_data, b_input_ids, b_attention_mask)
        loss = train_loss_fn(predictions, b_targets, b_nan_mask)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(train_loss_fn.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # ── Validate ──────────────────────────────────────────────────────
    model.eval()
    total_val_loss = 0
    all_preds, all_targets_list, all_masks = [], [], []

    with torch.no_grad():
        for batch_idx, batch in enumerate(val_loader):
            if batch_idx % 100 == 0:
                print(
                    f"\r  Epoch {epoch+1:02d}/{EPOCHS} | "
                    f"Val   batch {batch_idx}/{len(val_loader)}   ",
                    end="", flush=True,
                )

            graph_data      = batch['graph'].to(DEVICE)
            b_input_ids     = batch['input_ids'].to(DEVICE)
            b_attention_mask = batch['attention_mask'].to(DEVICE)
            b_targets       = batch['target'].to(DEVICE)
            b_nan_mask      = batch['nan_mask'].to(DEVICE)

            predictions = model(graph_data, b_input_ids, b_attention_mask)
            loss = masked_mse_loss(predictions, b_targets, b_nan_mask)
            total_val_loss += loss.item()

            all_preds.append(predictions.cpu())
            all_targets_list.append(b_targets.cpu())
            all_masks.append(b_nan_mask.cpu())

    avg_val_loss = total_val_loss / len(val_loader)
    scheduler.step(epoch + avg_val_loss / len(val_loader))

    # ── Per-property metrics (inverse-scaled) ─────────────────────────
    y_pred = torch.cat(all_preds, dim=0).numpy()
    y_true = torch.cat(all_targets_list, dim=0).numpy()
    y_mask = torch.cat(all_masks, dim=0).numpy()

    for i, col in enumerate(TARGET_COLS):
        if col in t_scalers:
            y_pred[:, i] = t_scalers[col].inverse_transform(y_pred[:, i].reshape(-1, 1)).flatten()
            y_true[:, i] = t_scalers[col].inverse_transform(y_true[:, i].reshape(-1, 1)).flatten()

    mae_per_prop, r2_per_prop = [], []
    for i in range(len(TARGET_COLS)):
        valid_idx = y_mask[:, i] == 1
        if valid_idx.sum() > 0:
            mae_per_prop.append(mean_absolute_error(y_true[valid_idx, i], y_pred[valid_idx, i]))
            r2_per_prop.append(r2_score(y_true[valid_idx, i], y_pred[valid_idx, i]))
        else:
            mae_per_prop.append(0.0)
            r2_per_prop.append(0.0)

    overall_mae = sum(mae_per_prop) / len(TARGET_COLS)
    overall_r2  = sum(r2_per_prop)  / len(TARGET_COLS)

    # ── Record history ────────────────────────────────────────────────
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_mae'].append(overall_mae)
    history['val_r2'].append(overall_r2)

    elapsed = time.time() - epoch_start

    # ── Epoch summary ─────────────────────────────────────────────────
    improved = avg_val_loss < best_val_loss
    marker = "🟢 NEW BEST" if improved else f"🔴 No improvement ({early_stop_counter+1}/{PATIENCE})"

    print(
        f"\r  Epoch {epoch+1:02d}/{EPOCHS} │ "
        f"Train: {avg_train_loss:.4f} │ Val: {avg_val_loss:.4f} │ "
        f"MAE: {overall_mae:.4f} │ R²: {overall_r2:.4f} │ "
        f"{elapsed:.0f}s │ {marker}"
    )

    # ── Per-property breakdown (first epoch, every 5 epochs, or on improvement)
    if epoch == 0 or (epoch + 1) % 5 == 0 or improved:
        print("    ┌────────────┬──────────────┬──────────────┐")
        print("    │  Property  │     MAE      │      R²      │")
        print("    ├────────────┼──────────────┼──────────────┤")
        for i in range(len(TARGET_COLS)):
            print(f"    │ {TARGET_COLS[i]:>10s} │ {mae_per_prop[i]:>12.4f} │ {r2_per_prop[i]:>12.4f} │")
        print("    └────────────┴──────────────┴──────────────┘")

    # ── Checkpointing & early stopping ────────────────────────────────
    if improved:
        best_val_loss = avg_val_loss
        output_dir = os.path.dirname(HYBRID_MODEL_OUTPUT_PATH)
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
        checkpoint = {
            'state_dict': model.state_dict(),
            'split_info': {
                'dataset_length': len(full_dataset),
                'train_size': train_size,
                'val_size': val_size,
                'test_size': test_size,
                'seed': SEED,
                'train_indices': list(train_dataset.indices),
                'val_indices': list(val_dataset.indices),
                'test_indices': list(test_dataset.indices),
            },
        }
        torch.save(checkpoint, HYBRID_MODEL_OUTPUT_PATH)
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    if early_stop_counter >= PATIENCE:
        print(f"\n⏹️  Early stopping at epoch {epoch+1}. Restoring best weights.")
        ckpt = torch.load(HYBRID_MODEL_OUTPUT_PATH, map_location='cpu')
        state = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
        model.load_state_dict(state)
        break

total_time = time.time() - training_start
print("=" * 90)
print(f"🏁 Training complete in {total_time/60:.1f} min  |  Best val loss: {best_val_loss:.4f}")
print(f"   Model saved to: {HYBRID_MODEL_OUTPUT_PATH}")

---
## 8 · Results Summary

Plot training curves and display the final per-property metrics.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss curves
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE curve
axes[1].plot(epochs_range, history['val_mae'], color='tab:orange', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Validation MAE (inverse-scaled)')
axes[1].grid(True, alpha=0.3)

# R² curve
axes[2].plot(epochs_range, history['val_r2'], color='tab:green', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('R²')
axes[2].set_title('Validation R² (inverse-scaled)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("📈 Training curves saved to training_curves.png")

In [ ]:
# ── Final per-property table ─────────────────────────────────────────
import pandas as pd

results_df = pd.DataFrame({
    'Property': TARGET_COLS,
    'MAE':  mae_per_prop,
    'R²':   r2_per_prop,
}).set_index('Property')

# Append overall row
results_df.loc['OVERALL'] = [overall_mae, overall_r2]

print("\n📊 Final Validation Metrics (inverse-scaled)")
display(results_df.style.format({'MAE': '{:.4f}', 'R²': '{:.4f}'}).set_caption('Per-Property Results'))

In [ ]:
# ── GPU memory summary (if available) ────────────────────────────────
if DEVICE.type == 'cuda':
    print("\n🖥️  GPU Memory Summary")
    print(f"   Allocated : {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"   Reserved  : {torch.cuda.memory_reserved()/1e9:.2f} GB")
    print(f"   Max alloc : {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
elif DEVICE.type == 'mps':
    print("\n🍎 MPS backend — no detailed memory stats available.")
else:
    print("\n💻 CPU run — no GPU memory to report.")

print(f"\n✅ All done! Model checkpoint: {HYBRID_MODEL_OUTPUT_PATH}")